*  DSC530-T301 Data Exploration and Analysis
*  Weeks 5 & 6 Coding Assignment
*  Ander (Trevor) McKee

## Setup and Environment Notes

This week's assignment instructions note that we have free choice of functions, so before each solution I use a markdown cell to document **which functions I chose and why**, and the code comments walk through each step.

**Environment note:** this notebook runs in the `pandas-book` conda environment that ships with the textbook (Python 3.8, **pandas 1.2.0**). The book-era syntax runs cleanly, and the pandas 2.x deprecation fixes from the week's supplemental notes do **not** apply — those changes only exist in pandas ≥ 2.0.

In [1]:
# Imports used throughout the assignment.
import pandas as pd
import numpy as np

# Confirm the environment.
print(f"pandas version: {pd.__version__}")
print(f"numpy version:  {np.__version__}")

pandas version: 1.2.0
numpy version:  1.19.5


In [2]:
# Load the two datasets used in Exercises 1-7.
quakes = pd.read_csv("exercises/earthquakes.csv")

# index_col="date" tells pandas to use the dates as row labels, and
# parse_dates=True means to translate the dates that are strings into real Datetime objects.
# This is necessary for exercise 7, which filters by year and quarter.
faang = pd.read_csv("exercises/faang.csv", index_col="date", parse_dates=True)

# Jupyter renders the last expression in a cell as an HTML table, so just state a dataframe.
pd.DataFrame(
    {"rows": [quakes.shape[0], faang.shape[0]],
     "columns": [quakes.shape[1], faang.shape[1]]},
    index=["earthquakes", "faang"],
)

,rows,columns
earthquakes,9332,6
faang,1255,6


# Chapter 4, Exercise 1

> With the `earthquakes.csv` file, select all the earthquakes in Japan with a magnitude of 4.9 or greater using the `mb` magnitude type.

**Function notes:** `query()` appears simpler than a boolean mask here. A mask would require using the dataframe name three times over, while query() treats the column names like variables. parsed_place is the cleaned country/region column derived from place. The raw place strings are descriptive text like "80km W of Honshu, Japan", so they can't be equality-matched against 'Japan'.

In [3]:
# Filter by all three conditions.
# time is stored as milliseconds since epoch, so unit="ms" is required --
# without it, to_datetime assumes nanoseconds and every date becomes 1970.
japan_mb = quakes.query(
    "parsed_place == 'Japan' and magType == 'mb' and mag >= 4.9"
).assign(time=lambda x: pd.to_datetime(x.time, unit="ms"))

# The boolean mask as an example.
# quakes[(quakes.parsed_place == 'Japan') & (quakes.magType == 'mb') & (quakes.mag >= 4.9)]

# Display the columns needed to verify each condition.
japan_mb[["parsed_place", "magType", "mag", "place", "time"]]

,parsed_place,magType,mag,place,time
1563,Japan,mb,4.9,"293km ESE of Iwo Jima, Japan",2018-10-08 05:45:32.250
2576,Japan,mb,5.4,"37km E of Tomakomai, Japan",2018-10-04 23:58:48.010
3072,Japan,mb,4.9,"15km ENE of Hasaki, Japan",2018-10-03 15:15:32.490
3632,Japan,mb,4.9,"53km ESE of Hitachi, Japan",2018-10-02 03:27:51.260


# Chapter 4, Exercise 2

> Create bins for each full number of earthquake magnitude (for instance, the first bin is (0,1], the second is (1,2], and so on) with the `ml` magnitude type and count how many are in each bin.

**Function notes:** `pd.cut()` is for binning a continuous variable into intervals. Its intervals are *right-closed* by default, which matches the bin definition in the problem (`np.histogram()` uses right-open bins and would put a magnitude of exactly 1.0 in the wrong bin). I then count with `value_counts()`, adding `sort_index()` because `value_counts()` sorts by frequency and the bins are easier to read by magnitude.

**Data notes:** 
- There are 504 ml readings at or below zero, 459 negative and 45 at exactly 0.0 (very small micro-earthquakes can register at or below zero on this scale).
- The textbook pads its bins out to (8,9] and shows three empty ones; I sized the edges from the data instead, so the counts match but the empty bins don't appear.

In [4]:
# Filter by magType, then pull out mag column as a series.
ml_mags = quakes.query("magType == 'ml'").mag

# Bin into right-closed unit intervals: (0,1], (1,2], ..., (5,6].
# Instead of guessing a range, just find the maximum reading, take the ceiling, and add 1.
mag_bins = pd.cut(ml_mags, bins=np.arange(0, np.ceil(ml_mags.max()) + 1))

# Count per bin and sort_index() by magnitude.
mag_bins.value_counts().sort_index()

(0.0, 1.0]    2207
(1.0, 2.0]    3105
(2.0, 3.0]     862
(3.0, 4.0]     122
(4.0, 5.0]       2
(5.0, 6.0]       1
Name: mag, dtype: int64

**My note:** Almost everything lands in (0,2] — small quakes vastly outnumber big ones.

# Chapter 4, Exercise 4

> Build a crosstab with the earthquake data between the `tsunami` column and the `magType` column. Rather than showing the frequency count, show the maximum magnitude that was observed for each combination. Put the magnitude type along the columns.

**Function notes:** `pd.crosstab()` stands for cross-tabulation, which typically refers to a frequency table of two categoricals. Setting values to mag and aggfunc to max fills each cell with the highest magnitude for that combination instead. `pd.crosstab()` actually calls `pivot_table()`, so it'd be reasonable to use a pivot table, but you'd still need to override the default (for pivot, it's mean).

**Data note:** NaN means that combination of tsunami and magnitude never occurred.

In [5]:
# Use crosstab as specified by the exercise.
pd.crosstab(
    index=quakes.tsunami,
    columns=quakes.magType,
    values=quakes.mag,
    aggfunc="max",
)

magType,mb,mb_lg,md,mh,ml,ms_20,mw,mwb,mwr,mww
tsunami,,,,,,,,,,
0,5.6,3.5,4.11,1.1,4.2,NaN,3.83,5.8,4.8,6.0
1,6.1,NaN,NaN,NaN,5.1,5.7,4.41,NaN,NaN,7.5


**My note:** The biggest magnitude in the table is the 7.5 under mww with a tsunami, which makes sense — mww is the scale used for major quakes.

# Chapter 4, Exercise 6

> Create a pivot table of the FAANG data that compares the stocks. Put the ticker in the rows and show the averages of the OHLC and volume traded data.

**Function notes:** 
- `.pivot_table()` does this in one call: `index="ticker"` puts each stock in its own row, and since `aggfunc` defaults to the mean,  every numeric column is averaged automatically. A `groupby("ticker").mean()` would give the same numbers but the exercise asks for a pivot table. 
- Then, you'd just need to adjust the order, because `pivot_table()` sorts the columns alphabetically.

In [6]:
# Simple pivot call because it defaults to mean.
faang_pivot = faang.pivot_table(index="ticker")

# Restore OHLC + volume order.
faang_pivot[["open", "high", "low", "close", "volume"]]

,open,high,low,close,volume
ticker,,,,,
AAPL,47.277859,47.748526,46.795877,47.263357,1.360803e+08
AMZN,1644.072709,1662.839839,1619.840519,1641.726176,5.648994e+06
FB,171.472948,173.613347,169.303148,171.510956,2.765860e+07
GOOG,1113.554101,1125.777606,1101.001658,1113.225134,1.741965e+06
NFLX,319.620558,325.219322,313.187330,319.290319,1.146962e+07


# Chapter 4, Exercise 7

> Calculate the z-scores for each numeric column of Amazon's data (ticker is AMZN) in Q4 2018 using `apply()`.

**Function notes:**
- **Selecting Q4 2018:** Since I parsed the dates into a `DatetimeIndex` at read time, `faang.loc["2018-Q4"]` selects the whole quarter with partial string indexing.
- **Z-scores with `apply()`:** `apply()` runs a function once per column, so `(x - x.mean()) / x.std()` standardizes every numeric column against its own mean and standard deviation.

**Data note (why values may differ from other packages):** pandas' `std()` uses the *sample* standard deviation (`ddof=1`, dividing by n−1), while `scipy.stats.zscore()` defaults to the *population* standard deviation (`ddof=0`, dividing by n). With the 63 trading days in Q4 2018, the results differ from scipy's by a factor of √(63/62) ≈ 1.008 (about 0.8%). I kept pandas' sample default because Q4 is being treated as a sample of the stock's behavior.

In [7]:
amzn_q4 = (
    # Partial string indexing on the DatetimeIndex.
    faang.loc["2018-Q4"]
    
    # Narrow to Amazon.
    .query("ticker == 'AMZN'")
    
    # Drop non-numeric column.
    .drop(columns="ticker")
)
print(f"{len(amzn_q4)} trading days in Q4 2018")

# x is the entire column, so subtracting the mean and dividing
# by (sample) standard deviation applies to every value at once.
amzn_q4_z = amzn_q4.apply(lambda x: (x - x.mean()) / x.std())
amzn_q4_z[["open","high","low","close","volume"]].head(10)

63 trading days in Q4 2018


,open,high,low,close,volume
date,,,,,
2018-10-01,2.337813,2.368006,2.502113,2.385848,-1.630411
2018-10-02,2.190795,2.227302,2.247433,2.155037,-0.861879
2018-10-03,2.068570,2.058955,2.139987,2.025489,-0.920345
2018-10-04,1.850048,1.819474,1.781561,1.722816,-0.126582
2018-10-05,1.642819,1.628173,1.554416,1.584748,-0.298771
2018-10-08,1.348850,1.435736,1.337840,1.408550,-0.072632
2018-10-09,1.255226,1.397931,1.483660,1.449753,-1.110557
2018-10-10,1.241193,1.127040,0.824506,0.646139,1.351658
2018-10-11,0.346454,0.393958,0.357894,0.395494,2.518992


In [8]:
# Check that every column of z-scores has
# mean ~0 and (sample) standard deviation exactly 1.
amzn_q4_z[["open", "high", "low", "close", "volume"]].agg(["mean", "std"]).round(4)

,open,high,low,close,volume
mean,-0.0,-0.0,-0.0,0.0,0.0
std,1.0,1.0,1.0,1.0,1.0


# Chapter 4, Exercise 10

The ECDC's *daily number of new reported cases of COVID-19 by country worldwide* dataset — a snapshot with data through September 19, 2020 (the assignment describes it as through September 18, but the file's index runs one day further). Each row is one country-day with the count of newly reported cases.

## Part 1: Prepare the data

**Function notes:**
- `pd.to_datetime(..., format="%d/%m/%Y")` converts to datetime specifically with **day-first** formatting; otherwise, pandas would mis-parse the dates or swap ambiguous ones.
- `set_index("date")` — later parts select by date and use idxmax(), and both need the dates in the index.
- `replace()` works on an entire dataframe, swapping each key with the associated value.
- `sort_index()` — to ensure the entries are chronologically ordered and not backwards.

In [9]:
covid = (
    pd.read_csv("exercises/covid19_cases.csv")
    
    # Convert to datetime with day-first formatting.
    .assign(date=lambda x: pd.to_datetime(x.dateRep, format="%d/%m/%Y"))
    
    # Index by date.
    .set_index("date")
    
    # Replace long with short names.
    .replace({
        "United_States_of_America": "USA",
        "United_Kingdom": "UK",
    })
    
    # Sort chronologically.
    .sort_index()
)

print(f"{covid.shape[0]:,} rows from {covid.index.min():%Y-%m-%d} to {covid.index.max():%Y-%m-%d}")
print(f'{len(covid.loc["2020-09-19"])} countries reported on 2020-09-19')
covid.head()

43,718 rows from 2019-12-31 to 2020-09-19
208 countries reported on 2020-09-19


,dateRep,day,month,year,cases,deaths,countriesAndTerritories,geoId,countryterritoryCode,popData2019,continentExp,Cumulative_number_for_14_days_of_COVID-19_cases_per_100000
date,,,,,,,,,,,,
2019-12-31,31/12/2019,31,12,2019,0,0,Belgium,BE,BEL,11455519.0,Europe,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Mexico,MX,MEX,127575529.0,America,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Ecuador,EC,ECU,17373657.0,America,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Russia,RU,RUS,145872260.0,Europe,NaN
2019-12-31,31/12/2019,31,12,2019,0,0,Netherlands,NL,NLD,17282163.0,Europe,NaN


## Part 2: For the five countries with the most cases (cumulative), find the day with the largest number of cases

**Function notes:**
- `groupby(...).cases.sum()` groups countries, selects cases, and sums them.
- `nlargest(5)` picks the top five directly rather than sorting and slicing.
- `idxmax()` returns the *index label* of the maximum, which *is* the date of the worst day since I used the datetimes as the index.
- I also show the case count on that day using `max()`.

In [10]:
# Cumulative cases per country; nlargest(5) keeps the top five.
top5 = covid.groupby("countriesAndTerritories").cases.sum().nlargest(5)
top5

countriesAndTerritories
USA       6724667
India     5308014
Brazil    4495183
Russia    1091186
Peru       756412
Name: cases, dtype: int64

In [11]:
# Select all days from top 5 countries, group them by country, and select only cases
top5_daily = covid[covid.countriesAndTerritories.isin(top5.index)].groupby(
    "countriesAndTerritories"
).cases

# Make a df using a dictionary: each key becomes a column.
# peak_day holds the idxmax() results (dates), peak_cases holds the max() results (counts).
pd.DataFrame({
    "peak_day": top5_daily.idxmax(),
    "peak_cases": top5_daily.max(),
})

,peak_day,peak_cases
countriesAndTerritories,,
Brazil,2020-07-30,69074
India,2020-09-17,97894
Peru,2020-08-17,10143
Russia,2020-07-18,12640
USA,2020-07-25,78427


## Part 3: 7-day average change in cases for the last week, for the Part 2 countries

**Function notes:**
- `groupby([...countries..., pd.Grouper(freq="1D")]).cases.sum()` then `unstack(0)` reshapes the data to one **row per day, one column per country** — the layout that time-series methods need.
- `diff()` computes the day-over-day **change** in new cases.
- `rolling(7).mean()` averages the daily changes over a 7-day window. The middle days cancel out when you add the changes up, so it works out to (cases today − cases 7 days ago) / 7. It's the net change over a week, spread across seven days.
- `last("1W")` grabs the last calendar week without computing the cutoff dates by hand. It was removed in pandas 2.x, but this notebook is pinned to 1.2.0.

In [12]:
(
    # Group by country and day, then sum to catch stray duplicates.
    covid.groupby(["countriesAndTerritories", pd.Grouper(freq="1D")]).cases.sum()
    
    # Long to wide format where rows = days and columns = countries.
    .unstack(0)
    
    # Subtract the previous row from each row, the day-over-day change in new cases.
    .diff()
    
    # Average the daily changes over 7 days — equivalent to (today − 7 days ago) / 7.
    .rolling(7).mean()
    
    # Grab just the last week.
    .last("1W")
    
    # Grab just the top 5 countries.
    [top5.index]
)

countriesAndTerritories,USA,India,Brazil,Russia,Peru
date,,,,,
2020-09-14,473.714286,181.285714,35.285714,36.285714,73.142857
2020-09-15,1513.000000,1142.857143,697.428571,46.285714,377.571429
2020-09-16,3478.714286,59.571429,3196.285714,61.428571,-65.000000
2020-09-17,-1047.000000,308.428571,143.428571,810.000000,-29.428571
2020-09-18,865.714286,-18.142857,-607.714286,-688.428571,-227.571429
2020-09-19,306.857143,-604.714286,-560.142857,57.285714,-41.285714


**My note:** Most of the values are negative, which means new cases were falling that week in most of the top five — the change can go both directions, so negative here is decline, not an error.

## Part 4: First date that each country other than China had cases

**Function notes:** 
- `pivot()` reshapes to one column per country with the dates as the index (I believe each country has exactly one row per day, so `pivot()` would throw an error if untrue while `pivot_table()` would take the mean).
- `drop()` would throw an error if it can't find "China", and better to fail loudly than quietly.
- Then `apply()` runs `x[x > 0].index.min()` on each column: filter that country's series to days with at least one case and take the minimum index (earliest date).
- `sort_values()` orders the countries chronologically.
- `rename()` cleans the underscores out of the country names for display.

In [13]:
(
    # Reshape to one column per country with the dates as the index.
    covid.pivot(columns="countriesAndTerritories", values="cases")
    
    # Remove China as a column.
    .drop(columns="China")
    
    # Filter each row by at least one case, then grab index
    .apply(lambda x: x[x > 0].index.min())
    
    # Sort chronologically.
    .sort_values()
    
    # Replace underscores with spaces.
    .rename(lambda x: x.replace("_", " "))
)

countriesAndTerritories
Thailand         2020-01-13
Japan            2020-01-15
South Korea      2020-01-20
Taiwan           2020-01-21
USA              2020-01-21
                    ...    
Yemen            2020-04-10
Western Sahara   2020-04-26
Tajikistan       2020-05-01
Comoros          2020-05-02
Lesotho          2020-05-15
Length: 209, dtype: datetime64[ns]

**My note:** Thailand comes first in mid-January, and the earliest dates are mostly countries near China, which is what you'd expect from how it spread.

## Part 5: Rank the countries by cumulative cases using percentiles

**Function notes:** 
- `pivot_table(..., aggfunc="sum")` totals the cases per country (cumulative cases), and transposing puts countries in the rows so they can be ranked.
- `rank(pct=True)` converts each country's rank into a **percentile** — the fraction of countries with that many cases or fewer — and `method="max"` assigns tied countries the highest rank in their group, so a percentile of 1.0 means 100% of countries have totals at or below it.
- Sorting descending puts the most-affected countries at the top; the USA's percentile of 1.0 confirms it has the most cumulative cases, consistent with Part 2.

In [14]:
(
    # Make a one-row pivot table with the sum of all cases for that country.
    covid.pivot_table(columns="countriesAndTerritories", values="cases", aggfunc="sum")
    
    # Transpose countries into rows.
    .T
    
    # Percentile rank of cumulative cases.
    .rank(pct=True, method="max")
    
    # Sort by rank (most-affected).
    .sort_values("cases", ascending=False)
    
    # Make names more readable again.
    .rename(lambda x: x.replace("_", " "))
)

,cases
countriesAndTerritories,
USA,1.000000
India,0.995238
Brazil,0.990476
Russia,0.985714
Peru,0.980952
...,...
Greenland,0.023810
Montserrat,0.019048
Falkland Islands (Malvinas),0.019048


## References

- Molin, S. (2021). *Hands-On Data Analysis with Pandas* (2nd ed.). Packt Publishing.
- pandas 1.2.0 documentation: https://pandas.pydata.org/pandas-docs/version/1.2.0/